# Data Fundamentals for AI Application Development

**Hands-on Lab | Oracle AI Database 26ai**

---

In this notebook you will:

1. Connect to Oracle AI Database 26ai and explore the Prism smart city dataset
2. Generate vector embeddings using an in-database ONNX model
3. Create an HNSW vector index and perform semantic search
4. Execute a single SQL query that combines relational JOINs, JSON dot notation, graph traversal, and vector search

**Estimated time:** 30 minutes (core) + 15 minutes (optional Hybrid Vector Search section)

**Prerequisites:** A running Oracle AI Database 26ai Free container with the Prism schema pre-loaded.

## The Prism Dataset

Prism uses a curated smart city dataset with seven districts and 28 infrastructure assets including bridges, substations, pipelines, sensors, communication towers, and more. The dataset also includes maintenance logs, inspection reports with findings, and connectivity relationships between assets. All of this data lives in a single Oracle database and is projected as relational rows, JSON documents, graph relationships, and vector embeddings, with no duplication and no synchronization overhead.

### Configuration

Update the values below to match your LiveLabs environment.

In [ ]:
# === CONFIGURATION - UPDATE THESE VALUES ===

DB_USER     = "your_username"
DB_PASSWORD = "your_password"
DB_DSN      = "localhost:1521/FREEPDB1"
ONNX_MODEL  = "DEMO_MODEL"

In [ ]:
# Install and import dependencies
import oracledb
import json
from IPython.display import HTML, display

# Return LOB columns as Python strings instead of LOB objects
oracledb.defaults.fetch_lobs = False

def show_table(headers, rows, max_width=80):
    """Display query results as a clean HTML table."""
    style_th = "border:1px solid #ddd; padding:6px 10px; background:#f4f4f4; text-align:left;"
    style_td = "border:1px solid #ddd; padding:6px 10px; white-space:pre-wrap;"
    h = '<table style="border-collapse:collapse; font-size:13px;">'
    h += "<tr>" + "".join(f"<th style=\"{style_th}\">{c}</th>" for c in headers) + "</tr>"
    for row in rows:
        h += "<tr>" + "".join(f"<td style=\"{style_td}\">{v}</td>" for v in row) + "</tr>"
    h += "</table>"
    display(HTML(h))

print("Dependencies loaded.")

### Connect to the Database

In [ ]:
# Connect to Oracle AI Database 26ai (thin mode - no Oracle Client needed)
conn = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN)
cursor = conn.cursor()

# Verify the connection
cursor.execute("SELECT banner FROM v$version WHERE ROWNUM = 1")
banner = cursor.fetchone()[0]
print(f"Connected successfully!\n{banner}")

---
## 1. Explore the Prism Data Model

The Prism schema is already loaded with sample data. Let's see what's here before we build on top of it.

In [ ]:
# List the Prism tables and their row counts
tables = [
    'DISTRICTS', 'INFRASTRUCTURE_ASSETS', 'OPERATIONAL_PROCEDURES',
    'MAINTENANCE_LOGS', 'INSPECTION_REPORTS', 'INSPECTION_FINDINGS',
    'ASSET_CONNECTIONS', 'DOCUMENT_CHUNKS'
]

rows = []
for t in tables:
    cursor.execute(f'SELECT COUNT(*) FROM {t}')
    count = cursor.fetchone()[0]
    rows.append((t, count))

show_table(['Table', 'Row Count'], rows)

### Relational Data

Infrastructure assets are stored in normalized relational tables with foreign key relationships to districts.

In [ ]:
# Peek at infrastructure assets with their district
cursor.execute("""
    SELECT a.asset_id, a.name, a.asset_type, a.status, d.name AS district
    FROM infrastructure_assets a
    JOIN districts d ON d.district_id = a.district_id
    ORDER BY a.asset_id
    FETCH FIRST 8 ROWS ONLY
""")

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

### JSON Data (Dot Notation)

Each asset has a `specifications` column stored as the native JSON data type. Different asset types carry different technical attributes (span length for a bridge, voltage rating for a substation, diameter for a pipeline). JSON dot notation lets us extract these fields inline with standard SQL.

In [ ]:
# Extract JSON fields using dot notation alongside relational columns
cursor.execute("""
    SELECT a.name,
           a.asset_type,
           a.specifications.spanLength_m.number()   AS span_length_m,
           a.specifications.loadCapacity_t.number()  AS load_capacity_t,
           a.specifications.material.string()        AS material
    FROM infrastructure_assets a
    WHERE a.asset_type = 'bridge'
""")

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

### Property Graph (SQL/PGQ)

Asset connectivity (which pipeline feeds which substation, which sensor monitors which bridge) is stored in the `ASSET_CONNECTIONS` table and projected as the `CITYPULSE_GRAPH` property graph. SQL/PGQ `GRAPH_TABLE` syntax lets us query relationships directly.

In [ ]:
# Query the property graph for asset connections
cursor.execute("""
    SELECT *
    FROM GRAPH_TABLE (citypulse_graph
        MATCH (a IS asset) -[c IS connected_to]-> (b IS asset)
        COLUMNS (a.name           AS from_asset,
                 c.connection_type AS relationship,
                 b.name           AS to_asset)
    )
    FETCH FIRST 10 ROWS ONLY
""")

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

---
## 2. Vector Embeddings with an In-Database ONNX Model

AI embedding models derive semantic meaning from text and output arrays of numbers called **vectors**. Semantically similar content produces vectors that are closer together in vector space.

### Embeddings Stay Inside the Database

Oracle 26ai can load ONNX embedding models directly into the database. This means vector search runs entirely inside Oracle and **your data never leaves the database**. No external API calls, no data egress, no network latency for embedding generation. The model sits right next to your data.

### Verify the ONNX Model

The ONNX embedding model (`DEMO_MODEL`) has been pre-loaded into the database. Let's confirm it is available and that our user has access to it.

In [ ]:
# Verify the ONNX model is loaded and accessible
cursor.execute("""
    SELECT model_name, mining_function, algorithm, creation_date
    FROM all_mining_models
    WHERE model_name = :model_name
""", {"model_name": ONNX_MODEL})

result = cursor.fetchone()
if result:
    show_table(['Model Name', 'Mining Function', 'Algorithm', 'Created'], [result])
else:
    print(f"WARNING: Model {ONNX_MODEL} not found. Check that it has been loaded.")

### Generate an Embedding

Let's pass a piece of text through the model and see what comes back.

In [ ]:
# Generate a single vector embedding
cursor.execute(f"""
    SELECT VECTOR_EMBEDDING({ONNX_MODEL} USING
               'safety incident at pump station' AS data) AS embedding
    FROM DUAL
""")
embedding = cursor.fetchone()[0]

# Get the dimension count
cursor.execute(f"""
    SELECT VECTOR_DIMS(
        VECTOR_EMBEDDING({ONNX_MODEL} USING
            'safety incident at pump station' AS data)
    ) AS dimensions
    FROM DUAL
""")
dims = cursor.fetchone()[0]

print(f"Total dimensions: {dims}")
print(f"\nFirst 200 characters of the vector:")
print(str(embedding)[:200], "...")

### Insert a New Maintenance Log and Vectorize It

The Prism dataset already has vector embeddings pre-loaded in the `DOCUMENT_CHUNKS` table. Rather than re-vectorize everything (which would take a while), let's see the full pipeline in action by inserting a single new maintenance log and then creating the vector embeddings for it.

The pipeline has three steps:
1. **Insert** the maintenance log into the relational table
2. **Chunk** the narrative text using `DBMS_VECTOR_CHAIN.UTL_TO_CHUNKS`
3. **Embed** each chunk using `VECTOR_EMBEDDING` and store in `DOCUMENT_CHUNKS`

In [ ]:
# Step 1: Insert a new maintenance log for Harbor Bridge

# Look up Harbor Bridge's asset_id
cursor.execute("SELECT asset_id FROM infrastructure_assets WHERE name = 'Harbor Bridge'")
harbor_bridge_id = cursor.fetchone()[0]

narrative = (
    "Detected unusual vibration patterns on the north support cable during "
    "routine sensor sweep. Frequency analysis suggests possible fatigue stress "
    "at the cable anchor point near the western abutment. Corrosion visible on "
    "three secondary cable clamps. Recommending detailed structural inspection "
    "within 48 hours and temporary load restriction to single-lane traffic."
)

# Clean up any previous runs: remove chunks and logs with this narrative
cursor.execute("""
    DELETE FROM document_chunks
    WHERE source_table = 'maintenance_logs'
      AND source_id IN (
          SELECT log_id FROM maintenance_logs
          WHERE narrative LIKE '%north support cable%'
            AND asset_id = :asset_id
      )
""", {'asset_id': harbor_bridge_id})
chunks_cleaned = cursor.rowcount

cursor.execute("""
    DELETE FROM maintenance_logs
    WHERE narrative LIKE '%north support cable%'
      AND asset_id = :asset_id
""", {'asset_id': harbor_bridge_id})
logs_cleaned = cursor.rowcount

if logs_cleaned > 0:
    print(f"Cleaned up {logs_cleaned} previous log(s) and {chunks_cleaned} chunk(s) from prior runs.")

# Insert and retrieve the generated log_id
log_id_var = cursor.var(int)
cursor.execute("""
    INSERT INTO maintenance_logs (asset_id, log_date, severity, narrative)
    VALUES (:asset_id, SYSDATE, :severity, :narrative)
    RETURNING log_id INTO :log_id
""", {
    'asset_id': harbor_bridge_id,
    'severity': 'warning',
    'narrative': narrative,
    'log_id': log_id_var
})
new_log_id = log_id_var.getvalue()[0]
print(f"Step 1 complete: Inserted maintenance log with log_id = {new_log_id}")

# Step 2: Chunk the narrative
chunk_params = json.dumps({
    "max": 1000, "overlap": 100, "split": "sentence", "normalize": "all"
})

cursor.execute("""
    SELECT et.column_value
    FROM TABLE(
        DBMS_VECTOR_CHAIN.UTL_TO_CHUNKS(:input_text, JSON(:chunk_params))
    ) et
""", {'input_text': narrative, 'chunk_params': chunk_params})
chunks_raw = cursor.fetchall()
print(f"Step 2 complete: Created {len(chunks_raw)} chunk(s)")

# Step 3: Embed each chunk and insert into DOCUMENT_CHUNKS
for idx, (chunk_value,) in enumerate(chunks_raw, start=1):
    # UTL_TO_CHUNKS returns JSON objects; extract the chunk_data field
    if isinstance(chunk_value, str):
        try:
            chunk_json = json.loads(chunk_value)
            chunk_text = chunk_json.get("chunk_data", chunk_value)
        except json.JSONDecodeError:
            chunk_text = chunk_value
    else:
        chunk_text = str(chunk_value)

    cursor.execute(f"""
        INSERT INTO document_chunks
            (source_table, source_id, chunk_seq, chunk_text, embedding)
        VALUES (
            :source_table, :source_id, :chunk_seq, :chunk_text,
            VECTOR_EMBEDDING({ONNX_MODEL} USING :chunk_text AS data)
        )
    """, {
        'source_table': 'maintenance_logs',
        'source_id': new_log_id,
        'chunk_seq': idx,
        'chunk_text': chunk_text
    })

conn.commit()
print(f"Step 3 complete: Embedded and stored {len(chunks_raw)} chunk(s) in DOCUMENT_CHUNKS.")

### Verify the New Vectors

Let's query `DOCUMENT_CHUNKS` for the log we just inserted and confirm the chunks and embeddings exist.

In [ ]:
# Verify the chunks and embeddings were created
cursor.execute("""
    SELECT dc.chunk_id,
           dc.chunk_seq,
           SUBSTR(dc.chunk_text, 1, 100) AS chunk_preview,
           VECTOR_DIMS(dc.embedding)     AS dimensions
    FROM document_chunks dc
    WHERE dc.source_table = 'maintenance_logs'
      AND dc.source_id = :log_id
    ORDER BY dc.chunk_seq
""", {'log_id': new_log_id})

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

---
## 3. Create a Vector Index

To perform fast approximate nearest neighbor (ANN) search over the embeddings, we create an **HNSW** (Hierarchical Navigable Small Worlds) vector index. HNSW builds a multi-layered in-memory graph structure that enables high-recall similarity search.

We use **cosine distance**, which measures the angle between vectors. It focuses on meaning rather than magnitude, so a short sentence and a long paragraph about the same topic can still be close.

In [ ]:
# Drop the vector index if it already exists (for re-runs)
cursor.execute("""
    SELECT index_name FROM user_indexes
    WHERE index_name = 'IDX_CHUNK_EMBEDDING'
""")
if cursor.fetchone():
    cursor.execute("DROP INDEX idx_chunk_embedding")
    print("Existing index dropped.")

# Create the HNSW vector index
cursor.execute("""
    CREATE VECTOR INDEX idx_chunk_embedding
        ON document_chunks(embedding)
        ORGANIZATION INMEMORY NEIGHBOR GRAPH
        DISTANCE COSINE
        WITH TARGET ACCURACY 95
""")
print("Vector index idx_chunk_embedding created.")

In [ ]:
# Verify the index
cursor.execute("""
    SELECT index_name, index_type, status
    FROM user_indexes
    WHERE index_name = 'IDX_CHUNK_EMBEDDING'
""")

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

---
## 4. Vector Search

Now let's search. We'll embed a natural language question and find the most semantically similar content across all maintenance logs, inspection reports, and inspection findings.

In [ ]:
# Semantic search: structural damage on Harbor Bridge
cursor.execute(f"""
    SELECT source_table,
           SUBSTR(chunk_text, 1, 120) AS chunk_preview,
           asset_name,
           district_name,
           ROUND(
               VECTOR_DISTANCE(embedding,
                   VECTOR_EMBEDDING({ONNX_MODEL} USING
                       'structural damage and corrosion on Harbor Bridge' AS data),
                   COSINE), 4
           ) AS distance
    FROM v_chunks_unified
    ORDER BY distance
    FETCH FIRST 5 ROWS ONLY
""")

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

**How to read the results:** Cosine distance closer to 0 means more semantically similar. Notice that the results are relevant even when the exact search words don't appear in the source text. That is the power of semantic search: it matches on *meaning*, not keywords.

Also notice the `SOURCE_TABLE` column. Results come from multiple content types (maintenance logs, inspection reports, inspection findings) because the unified view spans all of them.

In [ ]:
# Semantic search: different query to show breadth
cursor.execute(f"""
    SELECT source_table,
           SUBSTR(chunk_text, 1, 120) AS chunk_preview,
           asset_name,
           district_name,
           ROUND(
               VECTOR_DISTANCE(embedding,
                   VECTOR_EMBEDDING({ONNX_MODEL} USING
                       'equipment failures near water treatment causing environmental risk' AS data),
                   COSINE), 4
           ) AS distance
    FROM v_chunks_unified
    ORDER BY distance
    FETCH FIRST 5 ROWS ONLY
""")

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

---
## 5. The Unified Query: Four Data Shapes, One SQL Statement

In a polyglot persistence world, the query you're about to run would require four separate databases, four network hops, and a synchronization strategy. In Oracle 26ai, it's **one query, one transaction, zero syncing**.

This query will:
1. **Vector search** finds semantically similar maintenance and inspection content
2. **Relational JOINs** pull in asset details, district info, and inspection history
3. **JSON dot notation** extracts technical specifications from the JSON column
4. **Graph traversal** finds physically connected assets

### Query 1: Harbor Bridge (Table Output)

In [ ]:
# Unified query: relational + JSON + graph + vector in one statement
cursor.execute(f"""
    WITH vector_hits AS (
        SELECT source_table, source_id, chunk_text,
               asset_id, asset_name, district_name,
               VECTOR_DISTANCE(embedding,
                   VECTOR_EMBEDDING({ONNX_MODEL} USING
                       'maintenance issues and structural condition of Harbor Bridge' AS data),
                   COSINE) AS distance
        FROM v_chunks_unified
        ORDER BY distance
        FETCH FIRST 5 ROWS ONLY
    ),
    connected AS (
        SELECT gt.to_asset, gt.connection_type
        FROM GRAPH_TABLE (citypulse_graph
            MATCH (a IS asset) -[c IS connected_to]- (b IS asset)
            WHERE a.name = 'Harbor Bridge'
            COLUMNS (
                b.name           AS to_asset,
                c.connection_type AS connection_type
            )
        ) gt
    )
    SELECT vh.source_table                           AS source,
           SUBSTR(vh.chunk_text, 1, 100)             AS content_preview,
           ROUND(vh.distance, 4)                     AS vector_dist,
           ia.name                                   AS asset,
           d.name                                    AS district,
           ia.specifications.spanLength_m.number()   AS span_m,
           ia.specifications.loadCapacity_t.number() AS capacity_t,
           ia.specifications.material.string()       AS material,
           (SELECT LISTAGG(c.to_asset || ' (' || c.connection_type || ')', ', ')
                   WITHIN GROUP (ORDER BY c.to_asset)
            FROM connected c)                        AS connected_assets
    FROM vector_hits vh
    JOIN infrastructure_assets ia ON ia.asset_id = vh.asset_id
    JOIN districts d ON d.district_id = ia.district_id
    ORDER BY vh.distance
""")

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

### Query 1: Same Results as JSON

Oracle can also project those same results as a JSON document. Same query, different output shape.

In [ ]:
# The same unified query, but output as a JSON document
cursor.execute(f"""
    WITH vector_hits AS (
        SELECT source_table, source_id, chunk_text, asset_id,
               asset_name, district_name,
               VECTOR_DISTANCE(embedding,
                   VECTOR_EMBEDDING({ONNX_MODEL} USING
                       'maintenance issues and structural condition of Harbor Bridge' AS data),
                   COSINE) AS distance
        FROM v_chunks_unified
        ORDER BY distance
        FETCH FIRST 5 ROWS ONLY
    ),
    connected AS (
        SELECT gt.to_asset, gt.connection_type
        FROM GRAPH_TABLE (citypulse_graph
            MATCH (a IS asset) -[c IS connected_to]- (b IS asset)
            WHERE a.name = 'Harbor Bridge'
            COLUMNS (
                b.name           AS to_asset,
                c.connection_type AS connection_type
            )
        ) gt
    ),
    result_data AS (
        SELECT vh.source_table, vh.chunk_text, vh.distance,
               ia.name AS asset_name, d.name AS district,
               ia.specifications AS specs
        FROM vector_hits vh
        JOIN infrastructure_assets ia ON ia.asset_id = vh.asset_id
        JOIN districts d ON d.district_id = ia.district_id
    )
    SELECT JSON_SERIALIZE(
        JSON_OBJECT(
            'query' VALUE 'maintenance issues and structural condition of Harbor Bridge',
            'results' VALUE (
                SELECT JSON_ARRAYAGG(
                    JSON_OBJECT(
                        'source'   VALUE r.source_table,
                        'preview'  VALUE SUBSTR(r.chunk_text, 1, 150),
                        'distance' VALUE ROUND(r.distance, 4),
                        'asset'    VALUE r.asset_name,
                        'district' VALUE r.district,
                        'specs'    VALUE r.specs
                    ) ORDER BY r.distance
                ) FROM result_data r
            ),
            'connected_assets' VALUE (
                SELECT JSON_ARRAYAGG(
                    JSON_OBJECT(
                        'asset'           VALUE c.to_asset,
                        'connection_type'  VALUE c.connection_type
                    )
                ) FROM connected c
            )
        ) RETURNING JSON
    ) AS json_output
    FROM DUAL
""")

result = cursor.fetchone()[0]
print(json.dumps(result, indent=2))

**One database. One query. Four data shapes. Zero synchronization tax.**

### Query 2: Graph-Driven Discovery

This query flips the perspective. Instead of starting from a known asset, we start from critical incidents and use graph traversal to discover which *other* assets are connected to the ones that had problems. Then we pull their inspection records and specs to assess downstream risk.

In [ ]:
# Graph-driven discovery: start from critical incidents, traverse outward
cursor.execute(f"""
    WITH critical_assets AS (
        SELECT DISTINCT ia.name AS asset_name
        FROM maintenance_logs ml
        JOIN infrastructure_assets ia ON ia.asset_id = ml.asset_id
        WHERE ml.severity = 'critical'
    ),
    all_connections AS (
        SELECT *
        FROM GRAPH_TABLE (citypulse_graph
            MATCH (a IS asset) -[c IS connected_to]- (b IS asset)
            COLUMNS (
                a.name           AS from_asset,
                c.connection_type AS connection_type,
                b.name           AS to_asset,
                b.asset_type     AS to_type
            )
        )
    ),
    impacted AS (
        SELECT DISTINCT
               ac.from_asset, ac.connection_type,
               ac.to_asset, ac.to_type
        FROM all_connections ac
        JOIN critical_assets ca ON ca.asset_name = ac.from_asset
    ),
    latest_inspections AS (
        SELECT ir.asset_id, ir.overall_grade, ir.summary,
               ROW_NUMBER() OVER (
                   PARTITION BY ir.asset_id ORDER BY ir.inspect_date DESC
               ) AS rn
        FROM inspection_reports ir
    )
    SELECT imp.from_asset                               AS critical_asset,
           imp.connection_type                           AS relationship,
           imp.to_asset                                  AS connected_asset,
           imp.to_type                                   AS asset_type,
           ia.specifications.voltageRating_kv.number()   AS voltage_kv,
           ia.specifications.diameter_mm.number()        AS diameter_mm,
           ia.specifications.height_m.number()           AS height_m,
           li.overall_grade                              AS last_grade,
           SUBSTR(li.summary, 1, 100)                    AS inspection_summary
    FROM impacted imp
    JOIN infrastructure_assets ia ON ia.name = imp.to_asset
    LEFT JOIN latest_inspections li
        ON li.asset_id = ia.asset_id AND li.rn = 1
    ORDER BY imp.from_asset, imp.connection_type
""")

cols = [c[0] for c in cursor.description]
rows = cursor.fetchall()
if rows:
    show_table(cols, rows)
else:
    print("No critical-severity maintenance logs found.")
    print("Tip: Change the severity filter to 'warning' and re-run.")

---
## 6. OPTIONAL: Hybrid Vector Search (+15 minutes)

**This section is optional.** If you have time, it demonstrates how Oracle's Hybrid Vector Index combines lexical keyword search with semantic vector search in a single index.

### The Problem

Vector search excels at semantic similarity but can miss exact matches on asset identifiers, codes, or specific terminology. Lexical search nails exact matches but misses semantically related content that uses different wording.

Consider: *"What safety incidents involved Substation Gamma and what were the root causes?"*

This needs **semantic understanding** of "root causes" AND **exact matching** on "Substation Gamma." Neither search alone gets it right. Hybrid search combines both and fuses the scores.

### Create a Dedicated Hybrid Search Table

We create a separate table for this demo so the hybrid index has a clean text column to work with.

In [ ]:
# Drop hybrid search objects if they exist (for re-runs)
for stmt in [
    "DROP INDEX idx_hybrid_demo FORCE",
    "DROP TABLE hybrid_search_demo PURGE",
]:
    try:
        cursor.execute(stmt)
    except Exception:
        pass

try:
    cursor.execute("""
        BEGIN DBMS_VECTOR_CHAIN.DROP_PREFERENCE('prism_hybrid_pref'); END;
    """)
except Exception:
    pass

print("Cleanup complete (safe for re-runs).")

# Create and populate the hybrid search demo table
cursor.execute("""
    CREATE TABLE hybrid_search_demo (
        doc_id      NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        asset_name  VARCHAR2(200),
        severity    VARCHAR2(20),
        source_type VARCHAR2(50),
        content     VARCHAR2(4000) NOT NULL
    )
""")

# Populate from maintenance logs
cursor.execute("""
    INSERT INTO hybrid_search_demo (asset_name, severity, source_type, content)
    SELECT ia.name, ml.severity, 'maintenance_log', ml.narrative
    FROM maintenance_logs ml
    JOIN infrastructure_assets ia ON ia.asset_id = ml.asset_id
""")
log_count = cursor.rowcount

# Populate from inspection findings
cursor.execute("""
    INSERT INTO hybrid_search_demo (asset_name, severity, source_type, content)
    SELECT ia.name, inf.severity, 'inspection_finding', inf.description
    FROM inspection_findings inf
    JOIN inspection_reports ir ON ir.report_id = inf.report_id
    JOIN infrastructure_assets ia ON ia.asset_id = ir.asset_id
    WHERE inf.description IS NOT NULL
""")
finding_count = cursor.rowcount

conn.commit()
print(f"Table created and populated: {log_count} maintenance logs + {finding_count} inspection findings")

### Create the Vectorizer Preference and Hybrid Index

A **vectorizer preference** tells Oracle how to chunk and embed text inside the hybrid index. The hybrid index then builds both a full-text (Oracle Text) index and a vector index in one unified structure.

In [ ]:
# Create the vectorizer preference
cursor.execute("""
    BEGIN
        DBMS_VECTOR_CHAIN.CREATE_PREFERENCE(
            'prism_hybrid_pref',
            DBMS_VECTOR_CHAIN.VECTORIZER,
            JSON('{"vector_idxtype": "hnsw",
                   "model":          "DEMO_MODEL",
                   "by":             "words",
                   "max":            100,
                   "overlap":         10,
                   "split":          "recursively"}')
        );
    END;
""")
print("Vectorizer preference created.")

# Create the hybrid vector index (this may take a moment)
print("Creating hybrid vector index (this may take a moment)...")
cursor.execute("""
    CREATE HYBRID VECTOR INDEX idx_hybrid_demo
        ON hybrid_search_demo(content)
        PARAMETERS('VECTORIZER prism_hybrid_pref')
""")
print("Hybrid vector index idx_hybrid_demo created.")
print()
print("Note: The hybrid index uses MAINTENANCE AUTO, which means Oracle is")
print("chunking, embedding, and indexing all rows in the background.")
print("The first search query may take 30-60 seconds while this completes.")

### Vector-Only Search

First, let's see what a pure semantic search returns through the hybrid index.

> **Heads up:** The first query after index creation may take 30-60 seconds. The hybrid index is finishing its background synchronization (chunking, embedding, and indexing all rows). Subsequent queries will be fast.

In [ ]:
# Pure vector-only search via the hybrid index
cursor.execute("""
    SELECT JSON_SERIALIZE(
        DBMS_HYBRID_VECTOR.SEARCH(
            JSON('{
                "hybrid_index_name" : "IDX_HYBRID_DEMO",
                "search_scorer"     : "rsf",
                "search_fusion"     : "VECTOR_ONLY",
                "vector": {
                    "search_text"  : "root cause analysis of electrical failures",
                    "search_mode"  : "DOCUMENT",
                    "aggregator"   : "MAX",
                    "score_weight" : 1
                },
                "return": {
                    "values" : ["rowid", "score", "vector_score", "chunk_text"],
                    "topN"   : 5
                }
            }')
        ) RETURNING JSON
    ) FROM DUAL
""")

result = cursor.fetchone()[0]
print("=== Vector-Only Search ===")
print(json.dumps(result, indent=2))

### Text-Only Search

Now a pure keyword search for "Substation Gamma" through the same hybrid index.

In [ ]:
# Pure text-only (keyword) search via the hybrid index
cursor.execute("""
    SELECT JSON_SERIALIZE(
        DBMS_HYBRID_VECTOR.SEARCH(
            JSON('{
                "hybrid_index_name" : "IDX_HYBRID_DEMO",
                "search_scorer"     : "rsf",
                "search_fusion"     : "TEXT_ONLY",
                "text": {
                    "contains"     : "Substation AND Gamma",
                    "score_weight" : 1
                },
                "return": {
                    "values" : ["rowid", "score", "text_score", "chunk_text"],
                    "topN"   : 5
                }
            }')
        ) RETURNING JSON
    ) FROM DUAL
""")

result = cursor.fetchone()[0]
print("=== Text-Only Search ===")
print(json.dumps(result, indent=2))

### Hybrid Search: Best of Both Worlds

Now we combine both searches. The hybrid index fuses the keyword scores and semantic scores using Relative Score Fusion (RSF), giving us results that are both semantically relevant AND contain the exact terms we need.

In [ ]:
# Full hybrid search: semantic + keyword combined
cursor.execute("""
    SELECT JSON_SERIALIZE(
        DBMS_HYBRID_VECTOR.SEARCH(
            JSON('{
                "hybrid_index_name" : "IDX_HYBRID_DEMO",
                "search_scorer"     : "rsf",
                "search_fusion"     : "UNION",
                "vector": {
                    "search_text"  : "root cause analysis of electrical failures",
                    "search_mode"  : "DOCUMENT",
                    "aggregator"   : "MAX",
                    "score_weight" : 1
                },
                "text": {
                    "contains"     : "Substation AND Gamma",
                    "score_weight" : 1
                },
                "return": {
                    "values" : ["rowid", "score", "vector_score", "text_score", "chunk_text"],
                    "topN"   : 5
                }
            }')
        ) RETURNING JSON
    ) FROM DUAL
""")

result = cursor.fetchone()[0]
print("=== Hybrid Search (UNION with RSF) ===")
print(json.dumps(result, indent=2))

### Why is text_score always 0?

You probably noticed that every row in the UNION results has a `text_score` of 0. That's not a bug.

With `UNION` fusion, the result set includes rows from *either* the vector search OR the text search. The top 5 by combined score are all coming purely from the vector side because "root cause analysis of electrical failures" is semantically rich and scores high. Those particular rows don't contain the exact words "Substation" and "Gamma" in their content, so their `text_score` is 0. The text matches that *do* mention "Substation Gamma" are getting pushed below the top 5 because their vector relevance to "electrical failures" is lower.

You also may have noticed that `score` and `vector_score` are different even when `text_score` is 0. That's because RSF normalizes the raw scores before combining them. `vector_score` is the raw semantic similarity, while `score` is the normalized, fused result.

The fix: change the fusion to `INTERSECT`. INTERSECT only returns rows that appear in **both** the text search AND the vector search. That's the real power of hybrid search: find results that are semantically about electrical failures AND mention Substation Gamma by name.

### Hybrid Search with INTERSECT Fusion

In [ ]:
# Hybrid search with INTERSECT: only rows matching BOTH searches
cursor.execute("""
    SELECT JSON_SERIALIZE(
        DBMS_HYBRID_VECTOR.SEARCH(
            JSON('{
                "hybrid_index_name" : "IDX_HYBRID_DEMO",
                "search_scorer"     : "rsf",
                "search_fusion"     : "INTERSECT",
                "vector": {
                    "search_text"  : "root cause analysis of electrical failures",
                    "search_mode"  : "DOCUMENT",
                    "aggregator"   : "MAX",
                    "score_weight" : 1
                },
                "text": {
                    "contains"     : "Substation AND Gamma",
                    "score_weight" : 1
                },
                "return": {
                    "values" : ["rowid", "score", "vector_score", "text_score", "chunk_text"],
                    "topN"   : 5
                }
            }')
        ) RETURNING JSON
    ) FROM DUAL
""")

result = cursor.fetchone()[0]
print("=== Hybrid Search (INTERSECT with RSF) ===")
print(json.dumps(result, indent=2))

Now every row has both a non-zero `vector_score` AND a non-zero `text_score`. These are results that are semantically relevant to electrical failures and contain the words "Substation Gamma."

### Tuning the Balance: Weighting Text Higher

RSF lets you control how much influence each search type has on the final score. By increasing `score_weight` on the text side, we tell Oracle to favor results where the keyword match is strong. This is useful when exact terminology matters more than broad semantic similarity.

In [ ]:
# Hybrid search with heavier text weighting
cursor.execute("""
    SELECT JSON_SERIALIZE(
        DBMS_HYBRID_VECTOR.SEARCH(
            JSON('{
                "hybrid_index_name" : "IDX_HYBRID_DEMO",
                "search_scorer"     : "rsf",
                "search_fusion"     : "UNION",
                "vector": {
                    "search_text"  : "root cause analysis of electrical failures",
                    "search_mode"  : "DOCUMENT",
                    "aggregator"   : "MAX",
                    "score_weight" : 1
                },
                "text": {
                    "contains"     : "Substation OR Gamma",
                    "score_weight" : 5
                },
                "return": {
                    "values" : ["rowid", "score", "vector_score", "text_score", "chunk_text"],
                    "topN"   : 5
                }
            }')
        ) RETURNING JSON
    ) FROM DUAL
""")

result = cursor.fetchone()[0]
print("=== Hybrid Search (UNION, text weight = 5) ===")
print(json.dumps(result, indent=2))

With `score_weight` set to 5 on the text side (versus 1 on the vector side), results containing "Substation" or "Gamma" get a significant scoring boost. Compare the `text_score` and `vector_score` columns to see how the weighting shifts which results rise to the top. This is one of the "knobs" you can tune for your application without changing any code.

### Interpreting the Comparison

- **Vector-only** found semantically relevant electrical failure content, but may have included results from other assets entirely.
- **Text-only** found documents mentioning "Substation Gamma" by name, but missed semantically related content that described the same issues using different wording.
- **Hybrid** found the best of both: results that are semantically relevant to electrical failures AND associated with Substation Gamma. The fused score reflects both relevance dimensions.

This is exactly the scenario from the presentation: a query like *"What safety incidents involved asset CMP-2241 and what were the root causes?"* needs both types of search working together.

### Cleanup

In [ ]:
# Clean up the hybrid search demo objects
cursor.execute("DROP INDEX idx_hybrid_demo FORCE")
cursor.execute("DROP TABLE hybrid_search_demo PURGE")

# Remove the vectorizer preference
cursor.execute("""
    BEGIN
        DBMS_VECTOR_CHAIN.DROP_PREFERENCE('prism_hybrid_pref');
    END;
""")
conn.commit()
print("Hybrid search demo objects cleaned up.")

---
## 7. Summary and Next Steps

In this notebook you:

- **Connected** to Oracle AI Database 26ai and explored the Prism smart city dataset across relational tables, JSON columns, a property graph, and vector embeddings
- **Inserted** a new maintenance log and watched the vectorization pipeline create chunks and embeddings using an in-database ONNX model
- **Created** an HNSW vector index and performed semantic search across all content types
- **Combined** relational JOINs, JSON dot notation, graph traversal, and vector search in a single SQL query

All of this happened in **one database, with one query language (SQL), and zero synchronization overhead**.

### Further Reading

- [Chunking Strategies](https://bit.ly/oracle-chunking)
- [HNSW Vector Indexes](https://bit.ly/oracle-hnsw)
- [Oracle AI Vector Search Documentation](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/)

In [ ]:
# Close the database connection
cursor.close()
conn.close()
print("Connection closed. Lab complete!")